# Locy Use Case: Probabilistic Risk Scoring

Evaluate vendor reliability by combining independent quality signals with MNOR (noisy-OR failure) and MPROD (joint reliability).

This notebook uses **schema-first mode** (recommended): labels, edge types, and typed properties are defined before ingest.


## How To Read This Notebook

- Step 1 initializes an isolated local database.
- Step 2 defines schema (the recommended production path).
- Step 3 seeds a minimal graph for this use case.
- Step 4 declares Locy rules and query statements.
- Steps 5-6 evaluate and inspect command/query outputs.
- Step 7 tells you what to look for in the results.


## 1) Setup

Creates a temporary database directory so the example is reproducible and leaves no state behind.


In [ ]:
import os
import shutil
import tempfile
from pprint import pprint

import uni_db

DB_DIR = tempfile.mkdtemp(prefix="uni_locy_")
print("DB_DIR:", DB_DIR)

db = uni_db.Database.open(DB_DIR)


## 2) Define Schema (Recommended)

Define labels, property types, and edge types before inserting data.


In [ ]:
(
    db.schema()
    .label("Vendor")
        .property("name", "string")
    .done()
    .label("Component")
        .property("name", "string")
    .done()
    .label("QualitySignal")
        .property("name", "string")
        .property("pass_rate", "float64")
    .done()
    .edge_type("SUPPLIES", ["Vendor"], ["Component"])
    .done()
    .edge_type("HAS_SIGNAL", ["Component"], ["QualitySignal"])
    .done()
    .apply()
)

print('Schema created')


## 3) Seed Graph Data

Insert only the entities/relationships needed for this scenario so rule behavior stays easy to inspect.


In [ ]:
db.execute("CREATE (:Vendor {name: 'ReliaCorp'})")
db.execute("CREATE (:Vendor {name: 'QuickParts'})")
db.execute("CREATE (:Vendor {name: 'BudgetSupply'})")
db.execute("CREATE (:Component {name: 'Sensor'})")
db.execute("CREATE (:Component {name: 'Motor'})")
db.execute("CREATE (:Component {name: 'Controller'})")
db.execute("CREATE (:Component {name: 'Battery'})")
db.execute("CREATE (:QualitySignal {name: 'Thermal Test', pass_rate: 0.95})")
db.execute("CREATE (:QualitySignal {name: 'Vibration Test', pass_rate: 0.90})")
db.execute("CREATE (:QualitySignal {name: 'Voltage Tolerance', pass_rate: 0.85})")
db.execute("CREATE (:QualitySignal {name: 'Humidity Test', pass_rate: 0.92})")
db.execute("CREATE (:QualitySignal {name: 'Load Test', pass_rate: 0.88})")
db.execute("CREATE (:QualitySignal {name: 'EMC Test', pass_rate: 0.75})")
db.execute("CREATE (:QualitySignal {name: 'Cycle Life', pass_rate: 0.80})")
db.execute("CREATE (:QualitySignal {name: 'Drop Test', pass_rate: 0.70})")
db.execute("MATCH (v:Vendor {name:'ReliaCorp'}), (c:Component {name:'Sensor'}) CREATE (v)-[:SUPPLIES]->(c)")
db.execute("MATCH (v:Vendor {name:'ReliaCorp'}), (c:Component {name:'Motor'}) CREATE (v)-[:SUPPLIES]->(c)")
db.execute("MATCH (v:Vendor {name:'QuickParts'}), (c:Component {name:'Motor'}) CREATE (v)-[:SUPPLIES]->(c)")
db.execute("MATCH (v:Vendor {name:'QuickParts'}), (c:Component {name:'Controller'}) CREATE (v)-[:SUPPLIES]->(c)")
db.execute("MATCH (v:Vendor {name:'BudgetSupply'}), (c:Component {name:'Controller'}) CREATE (v)-[:SUPPLIES]->(c)")
db.execute("MATCH (v:Vendor {name:'BudgetSupply'}), (c:Component {name:'Battery'}) CREATE (v)-[:SUPPLIES]->(c)")
db.execute("MATCH (c:Component {name:'Sensor'}), (s:QualitySignal {name:'Thermal Test'}) CREATE (c)-[:HAS_SIGNAL]->(s)")
db.execute("MATCH (c:Component {name:'Sensor'}), (s:QualitySignal {name:'Vibration Test'}) CREATE (c)-[:HAS_SIGNAL]->(s)")
db.execute("MATCH (c:Component {name:'Motor'}), (s:QualitySignal {name:'Voltage Tolerance'}) CREATE (c)-[:HAS_SIGNAL]->(s)")
db.execute("MATCH (c:Component {name:'Motor'}), (s:QualitySignal {name:'Humidity Test'}) CREATE (c)-[:HAS_SIGNAL]->(s)")
db.execute("MATCH (c:Component {name:'Controller'}), (s:QualitySignal {name:'Load Test'}) CREATE (c)-[:HAS_SIGNAL]->(s)")
db.execute("MATCH (c:Component {name:'Controller'}), (s:QualitySignal {name:'EMC Test'}) CREATE (c)-[:HAS_SIGNAL]->(s)")
db.execute("MATCH (c:Component {name:'Battery'}), (s:QualitySignal {name:'Cycle Life'}) CREATE (c)-[:HAS_SIGNAL]->(s)")
db.execute("MATCH (c:Component {name:'Battery'}), (s:QualitySignal {name:'Drop Test'}) CREATE (c)-[:HAS_SIGNAL]->(s)")
print('Seeded graph data')


## 4) Locy Program

`CREATE RULE` defines derived relations. `QUERY ... WHERE ... RETURN ...` reads from those relations.


In [ ]:
program = r'''
CREATE RULE component_failure_risk AS
MATCH (c:Component)-[:HAS_SIGNAL]->(s:QualitySignal)
FOLD risk = MNOR(1.0 - s.pass_rate)
YIELD KEY c, risk

CREATE RULE vendor_reliability AS
MATCH (v:Vendor)-[:SUPPLIES]->(c:Component)
WHERE c IS component_failure_risk
FOLD reliability = MPROD(1.0 - risk)
YIELD KEY v, reliability

QUERY component_failure_risk RETURN c.name AS component, risk
QUERY vendor_reliability RETURN v.name AS vendor, reliability
'''
print(program)


## 5) Evaluate Locy Program

Run the program, then inspect materialization stats (`iterations`, `strata`, and executed queries).


In [ ]:
out = db.locy().evaluate(program)

print("Derived relations:", list(out.derived.keys()))
stats = out.stats
print("Iterations:", stats.total_iterations)
print("Strata:", stats.strata_evaluated)
print("Queries executed:", stats.queries_executed)


## 6) Inspect Command Results

Each command result can contain `rows`; this is the easiest way to verify your rule outputs and query projections.


In [ ]:
print("Derived relation snapshots:")
for rel_name, rel_rows in out.derived.items():
    print(f"\\n{rel_name}: {len(rel_rows)} row(s)")
    pprint(rel_rows)

if out.command_results:
    print("\\nCommand results:")
for i, cmd in enumerate(out.command_results, start=1):
    print(f"\\nCommand #{i}:", cmd.get("type"))
    rows = cmd.get("rows")
    if rows is not None:
        pprint(rows)
if not out.command_results:
    print("\\nNo QUERY/EXPLAIN/ABDUCE command outputs in this program.")


## 7) What To Expect

Use these checks to validate output after evaluation:
- Component risk ordering: Battery > Controller > Motor > Sensor (lower pass rates → higher risk).
- Vendor reliability ordering: ReliaCorp > QuickParts > BudgetSupply.
- MNOR values stay in [0, 1] — noisy-OR never exceeds 1.0 even with many signals.
- MPROD values decrease with more components — each additional component can only reduce joint reliability.
- Two query result blocks should appear: one for `component_failure_risk`, one for `vendor_reliability`.


## 8) Cleanup

Delete the temporary database directory created in setup.


In [ ]:
shutil.rmtree(DB_DIR, ignore_errors=True)
print("Cleaned up", DB_DIR)
